# 1 - Datasets and differential expression

Downloads the nine GEO series of the corpus, assigns every sample to the injury or the
control group, and compares the two groups within each series (Sections 4.1-4.2).

| Output | Use |
|---|---|
| `results/DE_<series>.csv` | one table per series, read by notebook 2 |
| `results/Table1_datasets.csv` | Table 1 |
| `results/geo_identification.csv`, `results/geo_corpus_coverage.csv`, `results/geo_identified_series.csv` | Figure 1 |
| `results/TableS2_screening.csv` | Table S2 |
| `results/TableS3_series.csv` | Table S3: input files, processing and model of each series |
| `results/TableS8_samples.csv` | Table S8: every analysed sample, with its GEO record and group |

Downloads are cached under `data/`, so a second run does not query GEO again.

In [1]:
# Move to the project root (the folder that contains run_all.ps1), so that relative
# paths work whether the notebook is started from notebook/ or from the root.
root <- normalizePath(getwd(), winslash = "/", mustWork = TRUE)
while (!file.exists(file.path(root, "run_all.ps1")) && dirname(root) != root) {
  root <- dirname(root)
}
if (!file.exists(file.path(root, "run_all.ps1"))) {
  stop("project root not found: no run_all.ps1 above ", getwd())
}
setwd(root)
.libPaths(c(file.path(root, "env", "rlib"), .libPaths()))
Sys.setenv(RENAL_IRI_ROOT = root)
options(stringsAsFactors = FALSE, warn = 1)
cat(R.version.string, "\nroot: ", root, "\n", sep = "")

R version 4.4.3 (2025-02-28)
root: /Users/robertocasale/Desktop/lavoro_FBF/renal_shield_perfusion


## Setup

Shared settings (`R/helpers.R`), and the dataset loaders with the differential-expression
function (`R/pipeline_de.R`).

In [2]:
source("R/helpers.R")
source("R/pipeline_de.R")

[2026-10-09 00:48:54] INFO  helpers.R loaded; constants and robust_fetch() ready.



## Identification

The GEO query of Section 4.1 and the number of series it returns. The answer is cached,
because the count can change over time.

In [3]:
n_identified <- geo_identification_search()
cov <- geo_corpus_coverage()
n_q <- sum(cov$returned_by_query[cov$role == "corpus"])
cat(sprintf("%d series returned by the declared query\n", n_identified))
cat(sprintf("%d of the %d analysed series are among them; %d were identified separately\n",
            n_q, length(REGISTRY), length(REGISTRY) - n_q))
print(cov)

[2026-10-09 00:49:02] INFO  CACHE HIT   geo_identification_726da479 (queried 2026-09-11 18:41:48)



[2026-10-09 00:49:02] INFO  GEO identification: 122 series match the declared query



[2026-10-09 00:49:02] INFO  CACHE HIT   geo_serieslist_726da479_retmax500_summary (queried 2026-09-13 17:43:13)



[2026-10-09 00:49:02] INFO  WROTE       results/geo_identified_series.csv (122 rows)



[2026-10-09 00:49:02] INFO  corpus coverage: 6 of 9 analysed series returned by the declared query; 3 identified separately



122 series returned by the declared query


6 of the 9 analysed series are among them; 3 were identified separately


   accession     role returned_by_query
1   GSE30718   corpus             FALSE
2   GSE43974   corpus             FALSE
3  GSE126805   corpus             FALSE
4   GSE39548   corpus              TRUE
5   GSE98622   corpus              TRUE
6   GSE34351   corpus              TRUE
7   GSE27274   corpus              TRUE
8   GSE58438   corpus              TRUE
9  GSE148420   corpus              TRUE
10 GSE267650 held-out              TRUE


## Screening (Table S2)

The decision for each of the series returned by the query: included in derivation,
reserved for external evaluation, eligible but not included, or excluded with the first
eligibility criterion of Section 4.1 not met. The decisions were taken from the GEO series
and sample records and are stored in `data/screening_decisions.csv`; the cell checks that
every series has one decision and that the analysed series carry the matching one.

In [4]:
# Table S2: the decision for every series returned by the query. The decisions were taken
# from the GEO series and sample records and are stored in data/screening_decisions.csv;
# this cell joins them to the query result and checks them against the analysed series.
listed  <- read.csv("results/geo_identified_series.csv")
decided <- read.csv("data/screening_decisions.csv", na.strings = character(0))
decisions <- c("Included in derivation", "Reserved for external evaluation",
               "Eligible, not included", "Excluded")
criteria  <- c("Kidney tissue profiling", "IRI-associated injury and reference comparison",
               "Whole-transcriptome platform", "Sample annotations sufficient for group assignment")
stopifnot(nrow(listed) == n_identified,
          !anyDuplicated(decided$accession),
          setequal(listed$accession, decided$accession),
          decided$decision %in% decisions,
          decided$criterion_not_met %in% c("", criteria),
          (decided$decision == "Excluded") == (decided$criterion_not_met != ""),
          setequal(decided$accession[decided$decision == "Included in derivation"],
                   cov$accession[cov$role == "corpus" & cov$returned_by_query]),
          setequal(decided$accession[decided$decision == "Reserved for external evaluation"],
                   cov$accession[cov$role == "held-out"]))
s2 <- merge(listed[, c("accession", "organism", "n_samples", "title")], decided, by = "accession")
s2 <- s2[order(match(s2$decision, decisions), match(s2$criterion_not_met, c("", criteria)),
               as.integer(sub("^GSE", "", s2$accession))), ]
names(s2) <- c("series", "organism", "samples", "geo_title", "decision", "criterion_not_met")
save_csv(s2, "TableS2_screening.csv")
cat(sprintf("%d series screened; %d met the eligibility criteria\n",
            nrow(s2), sum(s2$decision != "Excluded")))
print(setNames(as.vector(table(factor(s2$decision, decisions))), decisions))
cat("Excluded, by criterion not met:\n")
print(table(factor(s2$criterion_not_met[s2$decision == "Excluded"], criteria)))

[2026-10-09 00:49:02] INFO  WROTE       results/TableS2_screening.csv (122 rows)



122 series screened; 34 met the eligibility criteria


          Included in derivation Reserved for external evaluation 
                               6                                1 
          Eligible, not included                         Excluded 
                              27                               88 


Excluded, by criterion not met:



                           Kidney tissue profiling 
                                                55 
    IRI-associated injury and reference comparison 
                                                27 
                      Whole-transcriptome platform 
                                                 6 
Sample annotations sufficient for group assignment 
                                                 0 


## Differential expression

One table per series, with the gene symbols of the original platform; the mapping to
human orthologs is done in notebook 2. The column `sig` (BH-adjusted p < 0.05 and
|log2 fold change| >= 0.585) is descriptive only: notebook 2 ranks all genes by p-value.

In [5]:
de <- list(); summ <- list(); objs <- list()
for (acc in names(REGISTRY)) {
  obj <- REGISTRY[[acc]]()
  objs[[acc]] <- obj                       # kept for Tables S3 and S8 below
  d <- run_de(obj)
  d$accession <- acc; d$organism <- obj$organism; d$modality <- obj$modality
  de[[acc]] <- d
  save_csv(d, sprintf("DE_%s.csv", acc))
  summ[[acc]] <- data.frame(
    accession = acc, organism = obj$organism, platform = obj$platform,
    n = nrow(obj$meta), n_control = sum(obj$meta$group == "control"),
    n_IRI = sum(obj$meta$group == "IRI"), paired = isTRUE(obj$paired),
    genes = nrow(d), n_sig = sum(d$sig), contrast = obj$contrast)
  log_msg(sprintf("%-10s %-18s %6d genes, %5d flagged", acc, obj$organism, nrow(d), sum(d$sig)))
}
tab1 <- do.call(rbind, summ); rownames(tab1) <- NULL
save_csv(tab1, "Table1_datasets.csv")
print(tab1[, c("accession","organism","n","n_control","n_IRI","paired","genes","n_sig")],
      row.names = FALSE)

[2026-10-09 00:49:04] INFO  CACHE HIT   geo_probe_GSE30718 (queried 2026-09-11 18:42:10)



[2026-10-09 00:49:05] INFO  GSE30718: dropped 2 repeat biopsies from 2 patients (earliest kept per GEO)



[2026-10-09 00:49:06] INFO  WROTE       results/DE_GSE30718.csv (22880 rows)



[2026-10-09 00:49:06] INFO  GSE30718   Homo sapiens        22880 genes,   374 flagged



[2026-10-09 00:49:08] INFO  CACHE HIT   geo_probe_GSE43974 (queried 2026-09-11 18:43:27)



...

[2026-10-09 00:54:47] INFO  CACHE HIT   geo_probe_GSE148420 (queried 2026-09-11 18:49:17)



[2026-10-09 00:54:48] INFO  WROTE       results/DE_GSE148420.csv (19297 rows)



[2026-10-09 00:54:48] INFO  GSE148420  Rattus norvegicus   19297 genes,  3584 flagged



[2026-10-09 00:54:48] INFO  WROTE       results/Table1_datasets.csv (9 rows)



 accession          organism   n n_control n_IRI paired genes n_sig
  GSE30718      Homo sapiens  37        11    26  FALSE 22880   374
  GSE43974      Homo sapiens 332       163   169  FALSE 34694   179
 GSE126805      Homo sapiens  82        41    41   TRUE 16862   722
  GSE39548      Mus musculus   8         4     4  FALSE 14330  3629
  GSE98622      Mus musculus  21         6    15  FALSE 29325  1028
  GSE34351      Mus musculus   6         3     3  FALSE 21495  1049
  GSE27274 Rattus norvegicus  24         6    18  FALSE 21791  1866
  GSE58438 Rattus norvegicus  19         5    14  FALSE 18784  1074
 GSE148420 Rattus norvegicus  12         4     8  FALSE 19297  3584


## Series and samples (Tables S3 and S8)

Table S3 has one row per series: input files, processing reported by the submitters,
log2 transformation, column of the gene symbols, handling of duplicated symbols, gene
filter, model, selection of samples and numbers of samples. Table S8 has one row per
analysed sample (541): GEO accession and title, platform, group, GEO characteristics
(including tissue compartment and time point), and an identifier shared by the samples of
the same kidney (GSE126805, and the 96 kidneys with both biopsies in GSE43974) or the
same rat (GSE27274). No sample was excluded on quality grounds: the samples were selected
by design, as described in Table S3.

In [6]:
# Tables S3 and S8: how each series was read and processed (one row per series), and every
# sample analysed (one row per sample). They are built from the loader objects of the cell
# above (`objs`), from the GEO series matrix files and, for the kidney pairs of GSE43974,
# from the supplementary file of that series also used in notebook 2. The code stops if
# what Table S3 states does not match what the loaders and run_de() do.

# 1. GEO sample records: accession, title, platform, characteristics, description and the
#    submitters' description of their processing, read from the series matrix files (the
#    lines before the data table). The microarray files were downloaded by the loaders.
matrix_files <- list(
  GSE30718 = "GSE30718_series_matrix.txt.gz", GSE43974 = "GSE43974_series_matrix.txt.gz",
  GSE126805 = "GSE126805_series_matrix.txt.gz", GSE39548 = "GSE39548_series_matrix.txt.gz",
  GSE98622 = c("GSE98622-GPL13112_series_matrix.txt.gz", "GSE98622-GPL19057_series_matrix.txt.gz"),
  GSE34351 = "GSE34351_series_matrix.txt.gz", GSE27274 = "GSE27274_series_matrix.txt.gz",
  GSE58438 = "GSE58438_series_matrix.txt.gz", GSE148420 = "GSE148420_series_matrix.txt.gz")
stopifnot(setequal(names(matrix_files), names(objs)))
sample_records <- function(acc, file) {
  f <- file.path(DIRS$geo, file)
  if (!file.exists(f))
    utils::download.file(sprintf("https://ftp.ncbi.nlm.nih.gov/geo/series/%snnn/%s/matrix/%s",
                                 sub("[0-9]{3}$", "", acc), acc, file), f, mode = "wb", quiet = TRUE)
  con <- gzfile(f, "r"); on.exit(close(con))
  head <- character(0)
  repeat {
    l <- readLines(con, n = 1000)
    end <- grep("^!series_matrix_table_begin", l)
    head <- c(head, if (length(end)) l[seq_len(end[1] - 1)] else l)
    if (length(end) || !length(l)) break
  }
  field <- function(tag) lapply(grep(paste0("^!Sample_", tag, "\t"), head, value = TRUE),
                                function(x) gsub('"', "", strsplit(x, "\t")[[1]][-1]))
  out <- data.frame(series = acc, gsm = field("geo_accession")[[1]], geo_title = field("title")[[1]],
                    platform = field("platform_id")[[1]],
                    description = if (length(field("description"))) field("description")[[1]] else "",
                    data_processing = field("data_processing")[[1]])
  ch <- field("characteristics_ch1")           # "field: value"; a line can mix fields
  out$geo_characteristics <- vapply(seq_len(nrow(out)), function(i) {
    v <- vapply(ch, `[`, "", i); paste(v[v != ""], collapse = "; ") }, "")
  out
}
records <- do.call(rbind, lapply(names(matrix_files), function(acc) {
  r <- robust_fetch(paste0("geo_sample_records_", acc), fn = function()
    do.call(rbind, lapply(matrix_files[[acc]], function(f) sample_records(acc, f))))
  stopifnot(r$ok)
  r$value
}))

# 2. Kidney pairs of GSE43974 (array barcodes of the T2 and T3 biopsies of each kidney), from
#    the supplementary file of the series, with the same cache as notebook 2.
sheets <- robust_fetch("geo_suppl_GSE43974_paired_analyses_sheets", fn = function() {
  f  <- rownames(GEOquery::getGEOSuppFiles("GSE43974", baseDir = DIRS$geo,
                                           filter_regex = "paired_analyses"))
  xl <- sub("[.]gz$", "", f)
  R.utils::gunzip(f, xl, remove = FALSE, overwrite = TRUE)
  lapply(setNames(nm = readxl::excel_sheets(xl)),
         function(s) as.data.frame(readxl::read_excel(xl, sheet = s)))
})
stopifnot(sheets$ok)
pairs <- do.call(rbind, lapply(c("DBD", "DCD"), function(s) {
  x <- sheets$value[[s]]
  data.frame(kidney = sprintf("%s_%02d", s, seq_len(nrow(x))), T2 = x$T2, T3 = x$T3)
}))

# 3. Table S8: one row per analysed sample, with its GEO record. Microarray samples are named
#    by their GEO accession; RNA-seq samples by the sample title used in the deposited files.
#    `same_kidney_or_animal` is shared by the samples of one kidney (GSE126805, the kidneys
#    with both biopsies in GSE43974) or one rat (GSE27274).
species <- c("Homo sapiens" = "human", "Mus musculus" = "mouse", "Rattus norvegicus" = "rat")
kidney_of <- setNames(rep(pairs$kidney, 2), c(pairs$T2, pairs$T3))   # array barcode -> kidney
s8 <- do.call(rbind, lapply(names(objs), function(acc) {
  m <- objs[[acc]]$meta
  r <- records[records$series == acc, ]
  i <- ifelse(m$sample %in% r$gsm, match(m$sample, r$gsm), match(m$sample, r$geo_title))
  stopifnot(!anyNA(i), !anyDuplicated(i),
            all(r$platform[i] == sub(" .*$", "", objs[[acc]]$platform)))   # the platform of Table 1
  unit <- switch(acc,
    GSE126805 = m$subject,
    GSE27274  = m$animal,
    GSE43974  = unname(kidney_of[r$description[i]]),
    rep(NA_character_, nrow(m)))
  data.frame(series = acc, species = unname(species[objs[[acc]]$organism]), platform = r$platform[i],
             gsm = r$gsm[i], geo_title = r$geo_title[i],
             group = ifelse(m$group == "IRI", "injury", "reference"),
             same_kidney_or_animal = unit, geo_characteristics = r$geo_characteristics[i])
}))
t1 <- read.csv("results/Table1_datasets.csv")
n_ref <- tapply(s8$group == "reference", s8$series, sum)
n_inj <- tapply(s8$group == "injury", s8$series, sum)
k43 <- s8$series == "GSE43974" & !is.na(s8$same_kidney_or_animal)
stopifnot(nrow(s8) == sum(t1$n), !anyDuplicated(s8$gsm),
          all(n_ref[t1$accession] == t1$n_control), all(n_inj[t1$accession] == t1$n_IRI),
          sum(k43) == 2 * nrow(pairs), all(table(s8$same_kidney_or_animal[k43]) == 2),
          # each GSE43974 kidney pairs one reference (T2) and one injury (T3) biopsy
          all(tapply(s8$group[k43], s8$same_kidney_or_animal[k43], function(g) setequal(g, c("reference", "injury")))),
          all(table(s8$same_kidney_or_animal[s8$series %in% c("GSE126805", "GSE27274")]) == 2))
save_csv(s8, "TableS8_samples.csv")

# 4. Table S3: one row per series. Input files, the submitters' processing (GEO), whether
#    log2(x + 1) was applied (the rule of maybe_log2(), re-applied to the deposited values),
#    the column giving the gene symbols, the handling of duplicated symbols, the gene filter
#    and the model of run_de(), the selection of samples, and the numbers of samples.
input_files <- list(
  GSE126805 = "GSE126805_all.gene_counts.txt.gz (GEO supplementary file; raw gene counts)",
  GSE98622  = "GSE98622_mouse-iri-master.xlsx, Sheet1 (GEO supplementary file; processed expression values)")
symbol_column <- c(GSE30718 = "Gene Symbol", GSE43974 = "ILMN_Gene", GSE126805 = "external_gene_name",
                   GSE39548 = "GENE_SYMBOL", GSE98622 = "symbol", GSE34351 = "Gene Symbol",
                   GSE27274 = "ILMN_Gene", GSE58438 = "gene_assignment", GSE148420 = "GENE_SYMBOL")
selection <- c(
  GSE30718  = "Acute kidney injury biopsies (injury) and pristine protocol biopsies (reference); nephrectomy samples and later biopsies of the same patient (the earliest was kept) excluded",
  GSE43974  = "Deceased-donor biopsies at the end of cold ischemia (T2, reference) and after reperfusion (T3, injury); biopsies taken before organ retrieval (T1) and living-donor biopsies excluded",
  GSE126805 = "Biopsies before (reference) and after (injury) reperfusion of the kidneys with both; 3-month and 1-year biopsies, and post-reperfusion biopsies without a pre-reperfusion biopsy, excluded",
  GSE39548  = "Naive controls (reference) and IRI (injury), both without a protective manoeuvre; hemin-treated and ischemic preconditioning groups excluded",
  GSE98622  = "Sham at 4 and 24 h (reference) and IRI at 2, 4, 24, 48 and 72 h (injury); later time points (7 days to 12 months) and normal or aged controls excluded",
  GSE34351  = "Wild-type sham (reference) and ischemic (injury) kidneys, 4 h after reperfusion; TLR4-deficient mice excluded",
  GSE27274  = "Sham (reference) and IRI at 6, 24 and 120 h (injury), cortex and medulla of each rat; no sample excluded",
  GSE58438  = "Uninjured controls (reference) and saline-treated IRI at 3, 24 and 120 h (injury); valproic acid and dexamethasone groups excluded",
  GSE148420 = "Sham (reference) and unilateral IRI at 3 and 7 days (injury); no sample excluded")
stopifnot(setequal(names(symbol_column), names(objs)), setequal(names(selection), names(objs)))
xlsx98622 <- list.files(file.path(DIRS$geo, "GSE98622"), "xlsx$", full.names = TRUE)

s3 <- do.call(rbind, lapply(names(objs), function(acc) {
  obj <- objs[[acc]]
  if (acc == "GSE126805") {
    d <- read.delim(gzfile(file.path(DIRS$geo, "GSE126805", "GSE126805_all.gene_counts.txt.gz")),
                    check.names = FALSE)
    symbols <- as.character(d[[symbol_column[[acc]]]])
    logged <- NA
    files <- input_files[[acc]]
    duplicates <- "counts of rows with the same symbol summed"
  } else if (acc == "GSE98622") {
    d <- as.data.frame(suppressMessages(readxl::read_excel(xlsx98622[1], sheet = 1)))
    symbols <- as.character(d[[symbol_column[[acc]]]])
    logged <- attr(maybe_log2(as.matrix(d[, obj$meta$sample])), "logged")
    files <- input_files[[acc]]
    duplicates <- "row with the highest mean expression kept for each symbol"
  } else {
    es <- get_geo_es(acc)
    symbols <- first_symbol(Biobase::fData(es)[[symbol_column[[acc]]]])
    logged <- attr(maybe_log2(Biobase::exprs(es)), "logged")
    files <- paste0(paste(matrix_files[[acc]], collapse = "; "), "; ", obj$platform, ".soft.gz")
    duplicates <- "first symbol of probes annotated to several genes; probe with the highest mean expression kept for each symbol"
    stopifnot(file.exists(file.path(DIRS$geo, paste0(obj$platform, ".soft.gz"))))
  }
  # the stated symbol column must give exactly the genes of the loaded matrix
  stopifnot(setequal(rownames(obj$mat), setdiff(unique(symbols), c(NA, "", "---"))) ||
            (acc == "GSE126805" && setequal(rownames(obj$mat), unique(symbols))))
  model <- if (isTRUE(obj$is_counts) && isTRUE(obj$paired)) "DESeq2: counts ~ kidney + group" else
           if (!is.null(obj$covariate) && !is.null(obj$block))
             sprintf("limma: expression ~ %s + group; rat as a random block (duplicateCorrelation)", obj$covariate) else
           if (isTRUE(obj$rnaseq_continuous)) "limma with intensity trend: expression ~ group" else
           "limma: expression ~ group"
  r <- records[records$series == acc, ]
  done <- r$data_processing[match(s8$gsm[s8$series == acc], r$gsm)]
  data.frame(series = acc, species = unname(species[obj$organism]), platform = obj$platform,
             input_files = files,
             submitter_processing = paste(unique(done), collapse = " | "),
             log2_transformation = if (is.na(logged)) "not applicable (counts)" else
                                   if (logged) "log2(x + 1) applied (values on a linear scale)" else
                                   "none (values already on a log2 scale)",
             gene_symbols = sprintf("column %s", symbol_column[[acc]]),
             duplicated_symbols = duplicates,
             gene_filter = if (isTRUE(obj$is_counts)) "at least 10 counts in at least 20% of samples (minimum 2)" else
                           "values in at least 70% of samples and not all zero",
             model = model, sample_selection = selection[[acc]],
             samples_in_geo = nrow(r), reference = sum(obj$meta$group == "control"),
             injury = sum(obj$meta$group == "IRI"), genes_analysed = nrow(de[[acc]]))
}))
stopifnot(identical(s3$series, t1$accession), all(s3$reference == t1$n_control),
          all(s3$injury == t1$n_IRI), all(s3$genes_analysed == t1$genes))
save_csv(s3, "TableS3_series.csv")
print(s3[, c("series", "log2_transformation", "gene_symbols", "model", "samples_in_geo", "reference", "injury", "genes_analysed")],
      row.names = FALSE, right = FALSE)

[2026-10-09 00:54:48] INFO  CACHE HIT   geo_sample_records_GSE30718 (queried 2026-10-09 00:39:47)



[2026-10-09 00:54:48] INFO  CACHE HIT   geo_sample_records_GSE43974 (queried 2026-10-09 00:39:47)



[2026-10-09 00:54:48] INFO  CACHE HIT   geo_sample_records_GSE126805 (queried 2026-10-09 00:39:49)



[2026-10-09 00:54:48] INFO  CACHE HIT   geo_sample_records_GSE39548 (queried 2026-10-09 00:39:49)



[2026-10-09 00:54:48] INFO  CACHE HIT   geo_sample_records_GSE98622 (queried 2026-10-09 00:39:52)



...

[2026-10-09 00:55:01] INFO  CACHE HIT   geo_probe_GSE27274 (queried 2026-09-11 18:49:01)



[2026-10-09 00:55:02] INFO  CACHE HIT   geo_probe_GSE58438 (queried 2026-09-11 18:49:10)



[2026-10-09 00:55:03] INFO  CACHE HIT   geo_probe_GSE148420 (queried 2026-09-11 18:49:17)



[2026-10-09 00:55:03] INFO  WROTE       results/TableS3_series.csv (9 rows)



 series    log2_transformation                           
 GSE30718  none (values already on a log2 scale)         
 GSE43974  none (values already on a log2 scale)         
 GSE126805 not applicable (counts)                       
 GSE39548  none (values already on a log2 scale)         
 GSE98622  log2(x + 1) applied (values on a linear scale)
 GSE34351  log2(x + 1) applied (values on a linear scale)
 GSE27274  none (values already on a log2 scale)         
 GSE58438  log2(x + 1) applied (values on a linear scale)
 GSE148420 none (values already on a log2 scale)         
 gene_symbols             
 column Gene Symbol       
 column ILMN_Gene         
 column external_gene_name
 column GENE_SYMBOL       
 column symbol            
 column Gene Symbol       
 column ILMN_Gene         
 column gene_assignment   
 column GENE_SYMBOL       
 model                                                                           
 limma: expression ~ group                                          

Next: `02_analysis.ipynb`.